<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chess
import chess
import pandas as pd

# Ricaricho il dataset pulito
df = pd.read_csv('games_cleaned.csv')

# Conto quante mosse ci sono effettivamente nella stringa di testo da 'moves'

df['numero_mosse_reali'] = df['moves'].apply(lambda x: len(x.split()))
df = df[df['numero_mosse_reali'] >= 40].copy()

# Rimuovo la colonna di supporto per tenere il dataset pulito
df = df.drop(columns=['numero_mosse_reali'])
print(f"Partite lunghe almeno 40 semimosse: {df.shape[0]}")

def estrai_features_scacchiera(moves_string, n_semimosse):
    board = chess.Board()
    mosse = moves_string.split()

    for i in range(min(n_semimosse, len(mosse))):
        board.push_san(mosse[i])

    valori = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    mat_bianco = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.WHITE)
    mat_nero = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.BLACK)
    vantaggio_materiale = mat_bianco - mat_nero

    turno_originale = board.turn
    board.turn = chess.WHITE
    mobilita_bianco = board.legal_moves.count()
    board.turn = chess.BLACK
    mobilita_nero = board.legal_moves.count()
    board.turn = turno_originale
    vantaggio_mobilita = mobilita_bianco - mobilita_nero

    arrocco_bianco = int(board.has_castling_rights(chess.WHITE))
    arrocco_nero = int(board.has_castling_rights(chess.BLACK))

    # Re sotto scacco
    scacco = 1 if board.is_check() else 0

    return vantaggio_materiale, vantaggio_mobilita, arrocco_bianco, arrocco_nero, scacco

# 2. ESTRAZIONE MULTIPLA (10, 20, 30, 40 semimosse)
soglie = [10, 20, 30, 40]

for N in soglie:
    print(f"Calcolo feature per N={N} semimosse")
    risultati = df['moves'].apply(lambda x: estrai_features_scacchiera(x, N))

    # Creiamo 5 nuove colonne per ogni soglia temporale
    col_names = [f'mat_{N}', f'mob_{N}', f'arr_b_{N}', f'arr_n_{N}', f'scacco_{N}']
    df[col_names] = pd.DataFrame(risultati.tolist(), index=df.index)

# Salviamo il dataset completo e definitivo per la tesi
df.to_csv('games_features_complete.csv', index=False)
print("\nEstrazione completata. File 'games_features_complete.csv' salvato.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 8.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147878 sha256=45aa1b5bd9e2d3c065f5eed86d8a5aec56cf4d36a63d6a67bc492247b3a56eaf
  Stored in directory: /root/.cache/pip/wheels/74/65/99/8c3a88ba852c1ed8d34daed144b8cb4c654d638677483d19ff
Successfully built chess
Partite lunghe almeno 40 semimosse: 13007
Calcolo feature per N=10 semimosse... (potrebbe volerci qualche minuto)
Calcolo feature per N=20 semimosse... (potrebbe volerci qualche minuto)
Calcolo feature per N=30 semimosse... (potrebbe volerci qualche minuto)
Calcolo feature per N=40 semimosse... (potrebbe volerci qualche minuto)

Estrazione completata! File 'games_features_complete.csv' salvato.


In [ ]:
import pandas as pd

#Ricarico l'ultimo dataset salvato
df = pd.read_csv('games_features_complete.csv')

# Calcolo Differenza di Rating e Rating Medio
df['rating_diff'] = df['white_rating'] - df['black_rating']
df['rating_mean'] = (df['white_rating'] + df['black_rating']) / 2

#Apertura: Codice ECO raggruppato
#Uso il nome corretto della colonna: 'opening_eco'
df['eco_group'] = df['opening_eco'].astype(str).str[0]

#Cadenza (Tempo base della partita)
#Estraggo solo il tempo base (es. "10" da "10+0")
df['base_time_min'] = df['increment_code'].apply(lambda x: int(x.split('+')[0]) if '+' in str(x) else 0)

print("FEATURE PRE-PARTITA CALCOLATE")
print(df[['rating_diff', 'rating_mean', 'eco_group', 'base_time_min']].head())

# Salviamo il dataset finale con TUTTE le feature (posizionali e pre-partita)
df.to_csv('games_features_full.csv', index=False)
print("\nSalvataggio completato! Il file 'games_features_full.csv' è pronto per l'addestramento.")

--- FEATURE PRE-PARTITA CALCOLATE ---
   rating_diff  rating_mean eco_group  base_time_min
0           -4       1498.0         C              5
1          -15       1446.5         D             20
2           54       1496.0         C             30
3           47       1415.5         C             15
4          172       1295.0         B             10

Salvataggio completato! Il file 'games_features_full.csv' è pronto per l'addestramento.
